## jeu 1
Ici je présenterai le premier jeu de données, et répondrai aux questions joint en ordre. Les visualisation pertinentes seront présenté ainsi.

Commencons par charger les données.

In [1]:
import pandas as pd

df = pd.read_csv('plateforme_apprentissage.csv')
df

,id_apprenant,methode,niveau_initial,score_final,reussite,age,region,heures_etude
0,AP0001,B,intermédiaire,70.6,1,38,Québec,28.1
1,AP0002,B,débutant,52.5,0,32,Outaouais,21.9
2,AP0003,A,avancé,56.2,0,21,Hors Québec,23.5
3,AP0004,A,avancé,70.5,1,36,Outaouais,17.8
4,AP0005,B,débutant,81.7,1,33,Québec,29.3
...,...,...,...,...,...,...,...,...
2395,AP2396,B,débutant,49.3,0,39,Hors Québec,26.8
2396,AP2397,B,débutant,83.7,1,42,Montréal,22.9
2397,AP2398,A,intermédiaire,51.5,0,19,Hors Québec,22.8
2398,AP2399,B,débutant,51.1,0,26,Outaouais,18.3


### Passerelle vers le rapport LaTeX

Tout chiffre cité dans le rapport sort d'ici. On l'enregistre avec `val()`
ou `pct()`, puis `ecrire_valeurs()` les dépose dans `resultats/valeurs.tex`,
que le `.tex` relit à chaque compilation via `\val{clé}`. Les tableaux
passent par `ecrire_tableau()` et se chargent avec `\tbl{nom}`.

Aucun nombre n'est donc recopié à la main dans le rapport : il suffit de
réexécuter le notebook pour que le PDF suive.


In [2]:
from pathlib import Path

DOSSIER_RESULTATS = Path("resultats")
DOSSIER_RESULTATS.mkdir(exist_ok=True)

_valeurs = {}


def val(cle, valeur, dec=1, unite=""):
    """Enregistre un chiffre destiné au rapport et le renvoie inchangé."""
    if isinstance(valeur, str):
        texte = valeur
    elif dec == 0:
        texte = f"{round(valeur):d}"
    else:
        texte = f"{valeur:.{dec}f}".replace(".", ",")  # virgule décimale
    # Un « - » de clavier est un trait d'union : mauvais glyphe, et LaTeX
    # y autorise une coupure de ligne. On impose le vrai signe moins.
    if texte.startswith("-"):
        texte = "$-$" + texte[1:]
    _valeurs[cle.replace("_", "-")] = texte + unite
    return valeur


def pct(cle, proportion, dec=1):
    """Idem, pour une proportion entre 0 et 1 rendue en pourcentage."""
    return val(cle, 100 * proportion, dec=dec, unite=r"\,\%")


def ecrire_valeurs(nom="valeurs.tex"):
    """Écrit toutes les valeurs enregistrées dans resultats/valeurs.tex."""
    chemin = DOSSIER_RESULTATS / nom
    lignes = ["% Généré par le notebook -- ne pas éditer à la main.\n"]
    for cle, texte in sorted(_valeurs.items()):
        lignes.append(f"\\expandafter\\gdef\\csname val@{cle}\\endcsname{{{texte}}}\n")
    chemin.write_text("".join(lignes), encoding="utf-8")
    print(f"{len(_valeurs)} valeurs écrites dans {chemin}")


def ecrire_tableau(nom, tableau, **kwargs):
    """Dépose un tableau LaTeX (booktabs) que le rapport charge via \\tbl{nom}."""
    chemin = DOSSIER_RESULTATS / f"{nom.replace('_', '-')}.tex"
    kwargs.setdefault("float_format", "%.3f")
    chemin.write_text(tableau.to_latex(**kwargs), encoding="utf-8")
    print(f"tableau écrit dans {chemin}")


### Q1 Comparez le taux de réussite global des deux méthodes. Laquelle recommanderiez-vous à la direction de la plateforme?

Pour ceci nous allons séparer les deux méthodes, et trouver les taux de réussite pour chacune.
Je vais commencer par verifier que methode et reussite n'ont pas de nuls, ou valeurs abérantes/éronnées.
Ensuite je vais faire 2 DF, DF_A et DF_B.
En calculant la moyenne, nous aurons le taux de réussite.

In [3]:
df.methode.value_counts(dropna=False)

methode
B    1200
A    1200
Name: count, dtype: int64

In [4]:
df.reussite.value_counts(dropna=False)

reussite
1    1385
0    1015
Name: count, dtype: int64

In [5]:
df_a=df[df['methode'] == 'A']
df_b=df[df['methode'] == 'B']
taux_a= df_a.reussite.mean()
taux_b= df_b.reussite.mean()
print(f"le taux de réussite pour la méthode A est de {taux_a:.2%}")
print(f"le taux de réussite pour la méthode B est de {taux_b:.2%}")

le taux de réussite pour la méthode A est de 70.33%
le taux de réussite pour la méthode B est de 45.08%


In [6]:
# Les chiffres de Q1 partent vers le rapport.
pct("jeu1-taux-a", taux_a)
pct("jeu1-taux-b", taux_b)
val("jeu1-n-a", len(df_a), dec=0)
val("jeu1-n-b", len(df_b), dec=0)
val("jeu1-ecart", abs(100 * (taux_b - taux_a)), dec=1, unite=r"\,points")

ecrire_valeurs()


5 valeurs écrites dans resultats/valeurs.tex
